# PY300 — Outlier Treatment, Missing Values, and Normalization

Raw data is almost never clean. Before modelling or analysis, you must **treat outliers**, **handle missing values**, and **normalize/scale** features.

## Topics Covered
1. Outlier treatment strategies (remove, cap, transform)
2. Finding and treating missing values
3. Normalization — Min-Max Scaling
4. Standardization — Z-Score Scaling
5. Other scaling methods (Robust, Log, Box-Cox)

> **Perspective:** Data cleaning typically takes 60–80% of a data scientist's time. This is not wasted effort — garbage in, garbage out. A mediocre model on clean data beats a sophisticated model on dirty data.

---
## 1. Outlier Treatment Strategies

After *detecting* outliers (see notebook 02), you must decide what to do with them:

| Strategy | When to Use |
|---|---|
| **Remove** | Clearly erroneous data (typos, sensor glitches) |
| **Cap (Winsorize)** | Keep the data point but limit its influence |
| **Transform** | Apply log/sqrt to compress the range |
| **Keep** | Genuine extreme values important to the analysis |

In [ ]:
## Outlier treatment examples

import pandas as pd
import numpy as np

np.random.seed(42)
data = np.concatenate([np.random.normal(50, 10, 100), [150, 160, -20, -30]])
df = pd.DataFrame({'value': data})

Q1 = df['value'].quantile(0.25)
Q3 = df['value'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

print(f"Bounds: [{lower:.2f}, {upper:.2f}]")
print(f"Original shape: {df.shape[0]}")

# Strategy 1: REMOVE outliers
df_removed = df[(df['value'] >= lower) & (df['value'] <= upper)]
print(f"After removal : {df_removed.shape[0]}")

# Strategy 2: CAP (Winsorize) — clip to bounds
df_capped = df.copy()
df_capped['value'] = df_capped['value'].clip(lower=lower, upper=upper)
print(f"After capping : min={df_capped['value'].min():.2f}, max={df_capped['value'].max():.2f}")

# Strategy 3: LOG TRANSFORM — compress large values
df_positive = df[df['value'] > 0].copy()
df_positive['log_value'] = np.log1p(df_positive['value'])   # log(1+x) handles values near 0
print(f"Log transform : original range [{df_positive['value'].min():.1f}, {df_positive['value'].max():.1f}]")
print(f"              → log range [{df_positive['log_value'].min():.2f}, {df_positive['log_value'].max():.2f}]")

---
## 2. Finding and Treating Missing Values

Missing data appears as `NaN`, `None`, or empty strings. Common causes: sensor failures, optional survey fields, data merging mismatches.

In [ ]:
## Detecting missing values

df = pd.DataFrame({
    'name':   ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank'],
    'age':    [25, np.nan, 35, 40, np.nan, 28],
    'salary': [50000, 60000, np.nan, 80000, 55000, np.nan],
    'dept':   ['IT', 'HR', 'IT', None, 'Finance', 'HR']
})

print("=== Original Data ===")
print(df)

print("\n=== Missing Value Count ===")
print(df.isnull().sum())

print(f"\n=== Missing % ===")
print((df.isnull().sum() / len(df) * 100).round(1))

print(f"\nTotal cells   : {df.size}")
print(f"Missing cells : {df.isnull().sum().sum()}")
print(f"Missing %     : {df.isnull().sum().sum() / df.size * 100:.1f}%")

In [ ]:
## Treating missing values — different strategies

# Strategy 1: DROP rows with any missing value
df_dropped = df.dropna()
print(f"After dropna()   : {len(df_dropped)} rows (lost {len(df) - len(df_dropped)} rows)")

# Strategy 2: FILL with a constant
df_fill_const = df.copy()
df_fill_const['dept'] = df_fill_const['dept'].fillna('Unknown')
print(f"Dept after fill  : {df_fill_const['dept'].tolist()}")

# Strategy 3: FILL with mean (numeric columns)
df_fill_mean = df.copy()
df_fill_mean['age'] = df_fill_mean['age'].fillna(df_fill_mean['age'].mean())
df_fill_mean['salary'] = df_fill_mean['salary'].fillna(df_fill_mean['salary'].median())
print(f"Age after mean   : {df_fill_mean['age'].tolist()}")
print(f"Salary after med : {df_fill_mean['salary'].tolist()}")

# Strategy 4: FORWARD FILL (for time series) — carry last known value
df_ffill = df.copy()
df_ffill['age'] = df_ffill['age'].ffill()
print(f"Age after ffill  : {df_ffill['age'].tolist()}")

# Strategy 5: INTERPOLATE (for time series) — linear interpolation
df_interp = df.copy()
df_interp['age'] = df_interp['age'].interpolate()
print(f"Age interpolated : {df_interp['age'].tolist()}")

### Choosing a Missing Value Strategy

| Strategy | When to Use | Caution |
|---|---|---|
| **Drop rows** | < 5% missing, data is plentiful | Loses information |
| **Fill with mean/median** | Numeric, roughly symmetric | Reduces variance |
| **Fill with mode** | Categorical data | May amplify bias |
| **Forward/backward fill** | Time series, ordered data | Assumes continuity |
| **Interpolation** | Time series, smooth trends | Assumes linearity |
| **ML-based imputation** | Complex patterns | KNN, MICE (advanced) |

---
## 3. Normalization — Min-Max Scaling

Scales data to a fixed range, typically **[0, 1]**.

```
x_scaled = (x - min) / (max - min)
```

**When to use:** Neural networks, distance-based algorithms (KNN, K-Means), image data.

In [ ]:
## Min-Max Normalization

from sklearn.preprocessing import MinMaxScaler

df = pd.DataFrame({
    'age':       [22, 35, 45, 28, 60],
    'salary':    [30000, 55000, 80000, 42000, 120000],
    'experience': [1, 8, 15, 4, 30]
})

print("=== Original ===")
print(df)

# Manual Min-Max
df_manual = (df - df.min()) / (df.max() - df.min())
print("\n=== Manual Min-Max ===")
print(df_manual.round(3))

# Using sklearn MinMaxScaler
scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
print("\n=== sklearn MinMaxScaler ===")
print(df_scaled.round(3))

# Learning Point: Min-Max is sensitive to outliers — a single extreme
# value can compress all other values into a tiny range.

---
## 4. Standardization — Z-Score Scaling

Transforms data to have **mean = 0** and **std = 1**.

```
x_scaled = (x - mean) / std
```

**When to use:** Linear regression, logistic regression, SVM, PCA — algorithms that assume features are centered and similarly scaled.

In [ ]:
## Z-Score Standardization

from sklearn.preprocessing import StandardScaler

df = pd.DataFrame({
    'age':       [22, 35, 45, 28, 60],
    'salary':    [30000, 55000, 80000, 42000, 120000],
    'experience': [1, 8, 15, 4, 30]
})

# Manual Z-Score
df_manual = (df - df.mean()) / df.std()
print("=== Manual Z-Score ===")
print(df_manual.round(3))

# Using sklearn StandardScaler
scaler = StandardScaler()
df_standard = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
print("\n=== sklearn StandardScaler ===")
print(df_standard.round(3))

# Verify: mean ≈ 0, std ≈ 1
print(f"\nMeans: {df_standard.mean().round(3).tolist()}")
print(f"Stds : {df_standard.std().round(3).tolist()}")

---
## 5. Other Scaling Methods

In [ ]:
## Robust Scaling — uses median and IQR (resistant to outliers)

from sklearn.preprocessing import RobustScaler

df = pd.DataFrame({
    'salary': [30000, 55000, 80000, 42000, 120000, 500000]  # 500K is an outlier
})

robust = RobustScaler()
df['robust_scaled'] = robust.fit_transform(df[['salary']])

minmax = MinMaxScaler()
df['minmax_scaled'] = minmax.fit_transform(df[['salary']])

print(df.round(3))

# Notice: MinMax compresses 5 values into [0, 0.19] due to the outlier.
# Robust scaling handles this much better because it uses median/IQR.

In [ ]:
## Log and Box-Cox Transforms — for skewed distributions

from scipy import stats

# Simulated right-skewed data (e.g., income, website visits)
np.random.seed(42)
skewed = np.random.exponential(scale=5000, size=1000)

# Log transform
log_transformed = np.log1p(skewed)

# Box-Cox transform (data must be strictly positive)
boxcox_transformed, lambda_val = stats.boxcox(skewed + 1)  # +1 to ensure positive

print(f"Original   — Skew: {stats.skew(skewed):.2f}, Mean: {skewed.mean():.0f}")
print(f"Log        — Skew: {stats.skew(log_transformed):.2f}")
print(f"Box-Cox    — Skew: {stats.skew(boxcox_transformed):.2f}, lambda={lambda_val:.3f}")

# Box-Cox automatically finds the best power transform to make data normal.
# lambda ≈ 0 means log transform, lambda ≈ 0.5 means sqrt, lambda ≈ 1 means no transform.

---
## Scaling Methods Comparison

| Method | Formula | Range | Outlier Resistant? | Best For |
|---|---|---|---|---|
| **Min-Max** | (x-min)/(max-min) | [0, 1] | No | Neural networks, images |
| **Z-Score (Standard)** | (x-mean)/std | Unbounded | No | Linear models, PCA |
| **Robust** | (x-median)/IQR | Unbounded | Yes | Data with outliers |
| **Log Transform** | log(1+x) | Compressed | Yes | Right-skewed data |
| **Box-Cox** | Optimal power transform | Varies | Yes | Making data normal |
| **MaxAbs** | x/max(|x|) | [-1, 1] | No | Sparse data |

> **Critical Rule:** Always fit the scaler on **training data only**, then apply the same transformation to test data. Never fit on the full dataset — that leaks information from the test set into training (data leakage).